<a href="https://colab.research.google.com/github/maurocollin/bigdata-saude-niteroi/blob/main/T%C3%B3picos_de_Big_Data_em_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Análise de Indicadores de Saúde Municipal: Previsão de Demandas por Bairro em Niterói-RJ

In [9]:
!pip install geobr folium scipy  -q
!pip install pysus -q
from pysus.online_data import CNES
import os
import pandas as pd
import numpy as np
import geobr
import folium
from scipy import stats
import matplotlib.pyplot as plt



# caminho base (facilita manutenção depois)
caminho = '/content/drive/MyDrive/Colab Notebooks/DATA/'

print("Carregando arquivos...")

# leitura das bases
df_pop = pd.read_csv(f'{caminho}Agregados_por_bairros_basico_BR.csv', sep=';', encoding='latin1') #IBGE_2024
df_ubs = pd.read_csv(f'{caminho}Unidades_Básicas_de_Saúde_UBS.csv', encoding='utf-8')
df_hosp = pd.read_csv(f'{caminho}Hospitais.csv', encoding='utf-8')
df_poli = pd.read_csv(f'{caminho}Policlínicas.csv', encoding='utf-8')
# df_aten = pd.read_csv(f'{caminho}Area_de_Atendimento_UBS.csv', encoding='utf-8')

print("Arquivos ok.\n")

Carregando arquivos...
Arquivos ok.



In [33]:
# Filtrando colunas de interesse para estatística descritiva (infraestrutura e leitos)
colunas_infra = [col for col in df_st.columns if col.startswith(('QTLEIT', 'QTINST', 'LEITHOSP'))]

print("--- Estatísticas Descritivas: Infraestrutura de Saúde em Niterói ---")
if len(colunas_infra) > 0:
    # Garantindo que as colunas sejam numéricas para que o .describe() inclua 'max'
    df_infra_num = df_st[colunas_infra].apply(pd.to_numeric, errors='coerce').fillna(0)

    estatisticas = df_infra_num.describe().transpose()

    # Exibimos apenas colunas que possuem algum valor (max > 0)
    if 'max' in estatisticas.columns:
        display(estatisticas[estatisticas['max'] > 0])
    else:
        display(estatisticas)
else:
    display(df_st.describe())

--- Estatísticas Descritivas: Infraestrutura de Saúde em Niterói ---


,count,mean,std,min,25%,50%,75%,max
QTLEITP1,948.0,0.476793,4.931258,0.0,0.0,0.0,0.0,76.0
QTLEITP2,948.0,1.050633,9.483246,0.0,0.0,0.0,0.0,157.0
QTLEITP3,948.0,0.900844,6.671242,0.0,0.0,0.0,0.0,93.0
LEITHOSP,948.0,0.03481,0.183395,0.0,0.0,0.0,0.0,1.0
QTINST01,948.0,0.015823,0.253305,0.0,0.0,0.0,0.0,7.0
QTINST02,948.0,0.00211,0.045907,0.0,0.0,0.0,0.0,1.0
QTINST03,948.0,0.001055,0.032478,0.0,0.0,0.0,0.0,1.0
QTINST04,948.0,0.030591,0.305031,0.0,0.0,0.0,0.0,5.0
QTINST05,948.0,0.010549,0.112072,0.0,0.0,0.0,0.0,2.0
QTINST06,948.0,0.006329,0.11239,0.0,0.0,0.0,0.0,3.0


###### Tratamento da base de População do IBGE

In [10]:
print("Filtrando dados de Niterói...")
pop = df_pop[df_pop['NM_MUN'] == 'Niterói'].copy()

# limpeza de texto
pop['NM_BAIRRO'] = pop['NM_BAIRRO'].str.strip().str.upper()

# tratando população (vem com ponto de milhar)
pop['v0001'] = (
    pop['v0001']
    .astype(str)
    .str.replace('.', '', regex=False)
)

pop['v0001'] = pd.to_numeric(pop['v0001'], errors='coerce')

# renomeando pra algo legível
pop.rename(columns={'v0001': 'POPULACAO'}, inplace=True)

print(f"{len(pop)} bairros encontrados.\n")
display(pop[['NM_BAIRRO', 'POPULACAO']].head())


Filtrando dados de Niterói...
52 bairros encontrados.



,NM_BAIRRO,POPULACAO
8642,BADU,7055
8643,BALDEADOR,4335
8644,BARRETO,18699
8645,BOA VIAGEM,1944
8646,CACHOEIRA,2998


###### Tratando as bases de saúde


In [11]:
print("Padronizando nomes de bairros...")
# Função apara retirar acentos, espaços e colocar todos em caixa alta
def limpar_bairro(serie):
    return (
        serie
        .str.normalize('NFKD')
        .str.encode('ascii', errors='ignore')
        .str.decode('utf-8')
        .str.strip()
        .str.upper()
    )

df_ubs['NM_BAIRRO'] = limpar_bairro(df_ubs['bairro'])
df_hosp['NM_BAIRRO'] = limpar_bairro(df_hosp['tx_bairro'])
df_poli['NM_BAIRRO'] = limpar_bairro(df_poli['bairro'])


# contagem por bairro
qtd_ubs = df_ubs['NM_BAIRRO'].value_counts().reset_index(name='QTD_UBS')
qtd_hosp = df_hosp['NM_BAIRRO'].value_counts().reset_index(name='QTD_HOSP')
qtd_poli = df_poli['NM_BAIRRO'].value_counts().reset_index(name='QTD_POLI')


# juntando tudo
infra = qtd_ubs.merge(qtd_hosp, on='NM_BAIRRO', how='outer')
infra = infra.merge(qtd_poli, on='NM_BAIRRO', how='outer')

# tratando NaN
infra = infra.fillna(0).infer_objects(copy=False)

# criando colunas auxiliares
infra['QTD_BASICA'] = infra['QTD_UBS'] + infra['QTD_POLI']
infra['TOTAL_GERAL'] = infra['QTD_BASICA'] + infra['QTD_HOSP']

print("Infraestrutura consolidada.\n")
display(infra.sort_values('TOTAL_GERAL', ascending=False))


Padronizando nomes de bairros...
Infraestrutura consolidada.



,NM_BAIRRO,QTD_UBS,QTD_HOSP,QTD_POLI,QTD_BASICA,TOTAL_GERAL
1,BARRETO,1.0,4.0,1.0,2.0,6.0
5,FONSECA,1.0,4.0,1.0,2.0,6.0
2,CENTRO,2.0,3.0,1.0,3.0,6.0
13,PIRATININGA,1.0,1.0,1.0,2.0,3.0
16,VITAL BRAZIL,1.0,0.0,1.0,2.0,2.0
11,MARAVISTA,1.0,0.0,1.0,2.0,2.0
15,SANTA ROSA,0.0,2.0,0.0,0.0,2.0
10,LARGO DA BATALHA,1.0,0.0,1.0,2.0,2.0
4,ENGENHOCA,1.0,0.0,1.0,2.0,2.0
0,BAIRRO DE FATIMA,0.0,1.0,0.0,0.0,1.0


##### INTEGRAÇÃO FINAL

In [12]:
print("Integrando bases...")

# garantir padrão igual
pop['NM_BAIRRO'] = limpar_bairro(pop['NM_BAIRRO'])

final = pop.merge(infra, on='NM_BAIRRO', how='left')
final = final.fillna(0).infer_objects(copy=False)

# habitantes por unidade
final['HAB_POR_UBS_POLI'] = final['POPULACAO'] / final['QTD_BASICA']

# substitui infinito por NaN
final['HAB_POR_UBS_POLI'] = final['HAB_POR_UBS_POLI'].replace([np.inf, -np.inf], np.nan)

# agora sim arredonda e converte
final['HAB_POR_UBS_POLI'] = final['HAB_POR_UBS_POLI'].round().astype('Int64')

# estatísticas só onde existe atendimento
atendidos = final[final['QTD_BASICA'] > 0]

media = atendidos['HAB_POR_UBS_POLI'].mean()
desvio = atendidos['HAB_POR_UBS_POLI'].std()

# z-score
final['Z_SCORE_DEMANDA'] = np.where(
    final['QTD_BASICA'] > 0,
    ((final['HAB_POR_UBS_POLI'] - media) / desvio).round(2),
    np.nan
)

# evitar alertas chatos
pd.set_option('future.no_silent_downcasting', True)

# só pra exibir bonito
vis = final.fillna(0)

print(f"\nProcessamento finalizado ({len(final)} bairros).")
print(f"Média da cidade: {media:.0f} hab/unidade\n")

display(
    vis[['NM_BAIRRO', 'POPULACAO', 'QTD_BASICA', 'HAB_POR_UBS_POLI', 'Z_SCORE_DEMANDA']]
    .sort_values('QTD_BASICA', ascending=False)
)

Integrando bases...

Processamento finalizado (52 bairros).
Média da cidade: 8435 hab/unidade



,NM_BAIRRO,POPULACAO,QTD_BASICA,HAB_POR_UBS_POLI,Z_SCORE_DEMANDA
9,CENTRO,17938,3.0,5979,-0.42
2,BARRETO,18699,2.0,9350,0.15
12,ENGENHOCA,16700,2.0,8350,-0.01
15,FONSECA,46317,2.0,23158,2.49
32,PIRATININGA,18492,2.0,9246,0.14
49,MARAVISTA,10584,2.0,5292,-0.53
25,LARGO DA BATALHA,9637,2.0,4818,-0.61
29,MORRO DO ESTADO,3111,1.0,3111,-0.90
35,SANTA BARBARA,6609,1.0,6609,-0.31
3,BOA VIAGEM,1944,0.0,0,0.00


In [13]:
# 1. Padronização e Limpeza (Tratamento de Veracidade - Etapa 3)
# Ajustando as colunas de bairro de cada base (UBS, Hospitais, Policlínicas)
df_ubs['BAIRRO_LIMPO'] = df_ubs['bairro'].str.upper().str.strip()
df_hosp['BAIRRO_LIMPO'] = df_hosp['tx_bairro'].str.upper().str.strip()
df_poli['BAIRRO_LIMPO'] = df_poli['bairro'].str.upper().str.strip()

# 2. Identificar ONDE está cada unidade (Localização por Bairro)
# Criamos uma lista com os nomes das unidades em cada bairro
localizacao_detalhada = df_hosp.groupby('BAIRRO_LIMPO')['tx_nome'].apply(lambda x: ", ".join(x)).reset_index(name='HOSPITAIS')
localizacao_ubs = df_ubs.groupby('BAIRRO_LIMPO')['nome'].apply(lambda x: ", ".join(x)).reset_index(name='UBSs')

# 3. Contagem para Análise Estatística (Etapa 4 e 5)
ubs_count = df_ubs['BAIRRO_LIMPO'].value_counts().reset_index(name='QTD_UBS')
hosp_count = df_hosp['BAIRRO_LIMPO'].value_counts().reset_index(name='QTD_HOSP')
poli_count = df_poli['BAIRRO_LIMPO'].value_counts().reset_index(name='QTD_POLI')

# Fixando o Merge: Usamos 'BAIRRO_LIMPO' em vez de 'index'
df_mapa_saude = pd.merge(ubs_count, hosp_count, on='BAIRRO_LIMPO', how='outer')
df_mapa_saude = pd.merge(df_mapa_saude, poli_count, on='BAIRRO_LIMPO', how='outer').fillna(0)

# 4. Cálculo de Indicadores Totais
df_mapa_saude['Total_Equipamentos'] = df_mapa_saude['QTD_UBS'] + df_mapa_saude['QTD_HOSP'] + df_mapa_saude['QTD_POLI']

# 5. Análise Estatística Descritiva (Média e Desvio Padrão)
media = df_mapa_saude['Total_Equipamentos'].mean()
desvio_padrao = df_mapa_saude['Total_Equipamentos'].std()

# 6. Identificação de Outliers via Z-Score (Etapa 7)
df_mapa_saude['Z_Score'] = stats.zscore(df_mapa_saude['Total_Equipamentos'])

print(f"--- Estatísticas de Niterói ---")
print(f"Média de equipamentos: {media:.2f}")
print(f"Desvio Padrão (Dispersão): {desvio_padrao:.2f}\n")

# Exibindo os bairros com mais concentração (Outliers)
outliers = df_mapa_saude[df_mapa_saude['Z_Score'] > 1.5].sort_values('Total_Equipamentos', ascending=False)
print("Bairros Identificados como Polos de Saúde (Outliers):")
display(outliers[['BAIRRO_LIMPO', 'Total_Equipamentos', 'Z_Score']])

--- Estatísticas de Niterói ---
Média de equipamentos: 2.29
Desvio Padrão (Dispersão): 1.86

Bairros Identificados como Polos de Saúde (Outliers):


,BAIRRO_LIMPO,Total_Equipamentos,Z_Score
1,BARRETO,6.0,2.050475
2,CENTRO,6.0,2.050475
5,FONSECA,6.0,2.050475


In [14]:
# Exemplo de como classificar no Pandas
def categorizar_gestao(nome):
    nome = nome.upper()
    if 'MUNICIPAL' in nome or 'ESTADUAL' in nome or 'FEDERAL' in nome or 'UNIVERSITÁRIO' in nome:
        return 'PÚBLICO'
    else:
        return 'PRIVADO/FILANTRÓPICO'

df_hosp['CATEGORIA'] = df_hosp['tx_nome'].apply(categorizar_gestao)
# 1. Aplica a função para criar a nova coluna 'CATEGORIA'
df_hosp['CATEGORIA'] = df_hosp['tx_nome'].apply(categorizar_gestao)

# 2. Imprime as primeiras 5 linhas mostrando o Nome e a Categoria
print("--- Amostra da Classificação ---")
print(df_hosp[['tx_nome', 'CATEGORIA']])

--- Amostra da Classificação ---
                                              tx_nome             CATEGORIA
0                      Complexo Hospitalar de Niterói  PRIVADO/FILANTRÓPICO
1                        Hospital de Clínicas do Ingá  PRIVADO/FILANTRÓPICO
2                Hospital Universitário Antônio Pedro               PÚBLICO
3                  Hospital Municipal Carlos Tortelly               PÚBLICO
4                   Hospital Psiquiátrico de Jurujuba  PRIVADO/FILANTRÓPICO
5                        Hospital de Clinicas Alameda  PRIVADO/FILANTRÓPICO
6                           Hospital de Olhos Niteroi  PRIVADO/FILANTRÓPICO
7                                     Hospital Icarai  PRIVADO/FILANTRÓPICO
8                       Hospital Getulio Vargas Filho  PRIVADO/FILANTRÓPICO
9                         Hospital da Polícia Militar  PRIVADO/FILANTRÓPICO
10                    Hospital de Olhos Santa Beatriz  PRIVADO/FILANTRÓPICO
11                 Hospital de Clínicas São Sebastião  

In [15]:
try:
# --- TAREFA 2 E 3: LIMPEZA E VERACIDADE ---
  def localizar_coluna_bairro(df):
  # Procura por variações de nome de coluna que contenham 'BAIRRO'
    for col in df.columns:
      if 'BAIRRO' in col.upper():
        return col
    return None

# Normalizando e consolidando a contagem
  counts = []

  for nome, df_current in [('UBS', df_ubs), ('HOSPITAL', df_hosp), ('POLI', df_poli)]:
    col_bairro = localizar_coluna_bairro(df_current)
  if col_bairro:
    df_current[col_bairro] = df_current[col_bairro].str.upper().str.strip()
  count = df_current[col_bairro].value_counts().reset_index()
  count.columns = ['BAIRRO_LIMPO', f'QTD_{nome}']
  counts.append(count)

# Cruzamento de dados (Merge)
  df_indicadores = counts[0]
  for c in counts[1:]:
    df_indicadores = pd.merge(df_indicadores, c, on='BAIRRO_LIMPO', how='outer')

  df_indicadores = df_indicadores.fillna(0)
  df_indicadores['TOTAL_EQUIPAMENTOS'] = df_indicadores.iloc[:, 1:].sum(axis=1)

  # --- TAREFA 4: ANÁLISE ESTATÍSTICA (MÉDIA, DESVIO PADRÃO E OUTLIERS) ---
  print("--- 2. Processando Estatística Descritiva ---")

  # Cálculo das métricas solicitadas na estrutura
  media = df_indicadores['TOTAL_EQUIPAMENTOS'].mean()
  mediana = df_indicadores['TOTAL_EQUIPAMENTOS'].median()
  desvio_padrao = df_indicadores['TOTAL_EQUIPAMENTOS'].std()

  # Identificação de Outliers (Z-Score) para detectar anomalias
  df_indicadores['z_score'] = stats.zscore(df_indicadores['TOTAL_EQUIPAMENTOS'])

  print(f"Média de Equipamentos por Bairro: {media:.2f}")
  print(f"Desvio Padrão (Dispersão): {desvio_padrao:.2f}")

  # Relatório de Outliers (Bairros com oferta atípica)
  outliers = df_indicadores[abs(df_indicadores['z_score']) > 1.5]
  print("\n--- Relatório de Outliers (Anomalias de Demanda) ---")
  display(outliers[['BAIRRO_LIMPO', 'TOTAL_EQUIPAMENTOS', 'z_score']])

  # --- TAREFA: CARGA E VISUALIZAÇÃO (LOAD) ---
  print("--- 3. Gerando Visualização Final (Impacto Social) ---")

  # Mapa de Bairros via Geobr
  malha = geobr.read_neighborhood(year=2010)
  malha = malha[malha['code_muni'] == 3303302].copy()
  malha['name_neighborhood'] = malha['name_neighborhood'].str.upper().str.strip()

  # Unindo dados ao mapa
  mapa_final = malha.merge(df_indicadores, left_on='name_neighborhood', right_on='BAIRRO_LIMPO', how='left').fillna(0)

  # Criação do Mapa Folium
  m = folium.Map(location=[-22.8833, -43.1036], zoom_start=13, tiles='cartodbpositron')

  def style_function(feature):
    z = feature['properties']['z_score']
    if z > 1.5: color = '#9b59b6' # Roxo: Outlier (Alta concentração)
    elif z < -1: color = '#e74c3c' # Vermelho: Vazio Assistencial
    elif feature['properties']['TOTAL_EQUIPAMENTOS'] > 0: color = '#2ecc71' # Verde: Atendimento Normal
    else: color = '#bdc3c7' # Cinza: Sem dados
    return {'fillColor': color, 'color': 'white', 'weight': 1, 'fillOpacity': 0.7}

    folium.GeoJson(mapa_final, style_function=style_function,
  tooltip=folium.GeoJsonTooltip(fields=['name_neighborhood', 'TOTAL_EQUIPAMENTOS', 'z_score'],
  aliases=['Bairro:', 'Equipamentos:', 'Z-Score:'])
  ).add_to(m)

  display(m)
  print("Projeto Concluído: Saneamento, Estatística e Carga finalizados.")
except Exception as e:
  print(f"Erro na conclusão: {e}")

--- 2. Processando Estatística Descritiva ---
Média de Equipamentos por Bairro: 1.00
Desvio Padrão (Dispersão): 0.00

--- Relatório de Outliers (Anomalias de Demanda) ---


/tmp/ipykernel_1879/2914212035.py:38: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  df_indicadores['z_score'] = stats.zscore(df_indicadores['TOTAL_EQUIPAMENTOS'])


,BAIRRO_LIMPO,TOTAL_EQUIPAMENTOS,z_score


--- 3. Gerando Visualização Final (Impacto Social) ---


Projeto Concluído: Saneamento, Estatística e Carga finalizados.


In [29]:

def coletar_dados_cnes_corrigido(ano, mes):
    codigo_niteroi = '330330'

    try:
        print(f"--- Coletando dados de {mes}/{ano} ---")

        # 1. Download dos Estabelecimentos (ST)
        print("Buscando Estabelecimentos (ST)...")
        dataset_st = CNES.download(states='RJ', years=ano, months=mes, group='ST')
        df_st_full = dataset_st.to_dataframe()

        # 2. Download de Leitos (LT)
        print("Buscando Leitos (LT)...")
        dataset_lt = CNES.download(states='RJ', years=ano, months=mes, group='LT')
        df_lt_full = dataset_lt.to_dataframe()

        # 3. Filtragem para Niterói
        print("Filtrando dados para Niterói...")
        # Garantindo que CODUFMUN seja tratado como string para o filtro
        df_st_nit = df_st_full[df_st_full['CODUFMUN'].astype(str).str.contains(codigo_niteroi)].copy()
        df_lt_nit = df_lt_full[df_lt_full['CODUFMUN'].astype(str).str.contains(codigo_niteroi)].copy()

        # 4. Salvar resultado final no Drive
        caminho_saida = '/content/drive/MyDrive/Colab Notebooks/DATA/'
        os.makedirs(caminho_saida, exist_ok=True)

        df_st_nit.to_csv(f'{caminho_saida}CNES_ST_Niteroi.csv', index=False)
        df_lt_nit.to_csv(f'{caminho_saida}CNES_LT_Niteroi.csv', index=False)

        print(f"✅ Processo concluído! ST: {len(df_st_nit)} linhas, LT: {len(df_lt_nit)} linhas.")
        return df_st_nit, df_lt_nit

    except Exception as e:
        print(f"❌ Erro: {e}")
        return None, None

# Execução
df_st, df_lt = coletar_dados_cnes_corrigido(2024, 1)

--- Coletando dados de 1/2024 ---
Buscando Estabelecimentos (ST)...


1416490it [00:00, 1793837461.64it/s] 


Buscando Leitos (LT)...


45717it [00:00, 70992593.84it/s]     

Filtrando dados para Niterói...


✅ Processo concluído! ST: 948 linhas, LT: 190 linhas.


In [34]:
# Verificando as colunas e os primeiros registros de df_st
if 'df_st' in globals() and df_st is not None:
    print(f"Colunas disponíveis: {df_st.columns.tolist()}\n")
    display(df_st)
else:
    print("O DataFrame 'df_st' não foi encontrado. Certifique-se de que a coleta anterior foi executada com sucesso.")

Colunas disponíveis: ['CNES', 'CODUFMUN', 'COD_CEP', 'CPF_CNPJ', 'PF_PJ', 'NIV_DEP', 'CNPJ_MAN', 'COD_IR', 'REGSAUDE', 'MICR_REG', 'DISTRSAN', 'DISTRADM', 'VINC_SUS', 'TPGESTAO', 'ESFERA_A', 'RETENCAO', 'ATIVIDAD', 'NATUREZA', 'CLIENTEL', 'TP_UNID', 'TURNO_AT', 'NIV_HIER', 'TP_PREST', 'CO_BANCO', 'CO_AGENC', 'C_CORREN', 'CONTRATM', 'DT_PUBLM', 'CONTRATE', 'DT_PUBLE', 'ALVARA', 'DT_EXPED', 'ORGEXPED', 'AV_ACRED', 'CLASAVAL', 'DT_ACRED', 'AV_PNASS', 'DT_PNASS', 'GESPRG1E', 'GESPRG1M', 'GESPRG2E', 'GESPRG2M', 'GESPRG4E', 'GESPRG4M', 'NIVATE_A', 'GESPRG3E', 'GESPRG3M', 'GESPRG5E', 'GESPRG5M', 'GESPRG6E', 'GESPRG6M', 'NIVATE_H', 'QTLEITP1', 'QTLEITP2', 'QTLEITP3', 'LEITHOSP', 'QTINST01', 'QTINST02', 'QTINST03', 'QTINST04', 'QTINST05', 'QTINST06', 'QTINST07', 'QTINST08', 'QTINST09', 'QTINST10', 'QTINST11', 'QTINST12', 'QTINST13', 'QTINST14', 'URGEMERG', 'QTINST15', 'QTINST16', 'QTINST17', 'QTINST18', 'QTINST19', 'QTINST20', 'QTINST21', 'QTINST22', 'QTINST23', 'QTINST24', 'QTINST25', 'QTINST2

,CNES,CODUFMUN,COD_CEP,CPF_CNPJ,PF_PJ,NIV_DEP,CNPJ_MAN,COD_IR,REGSAUDE,MICR_REG,...,AP07CV02,AP07CV03,AP07CV04,AP07CV05,AP07CV06,AP07CV07,ATEND_PR,DT_ATUAL,COMPETEN,NAT_JUR
8580,0012505,330330,24033900,15126437003673,3,1,00000000000000,,,,...,0,0,0,0,0,0,1,202401,202401,2011
8581,0012513,330330,24070090,32556060002397,3,3,32556060000181,,,,...,0,0,0,0,0,0,1,202312,202401,1155
8582,0012521,330330,24355090,42498717000660,3,3,42498717000155,,,,...,0,0,0,0,0,0,1,202311,202401,1023
8583,0012556,330330,24111000,32556060003369,3,3,32556060000181,,,,...,0,0,0,0,0,0,1,202402,202401,1155
8584,0012564,330330,24020070,32556060002559,3,3,32556060000181,,,,...,0,0,0,0,0,0,1,202312,202401,1155
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9523,9973877,330330,24130081,00000000000000,3,3,34906284000100,,,,...,0,0,0,0,0,0,1,202401,202401,1279
9524,9974067,330330,24230101,21507365000187,3,1,00000000000000,,,,...,0,0,0,0,0,0,1,202401,202401,2062
9525,9989048,330330,24220001,32147559000135,3,1,00000000000000,,,,...,0,0,0,0,0,0,1,202309,202401,2232
9526,9992413,330330,24020081,09118164000192,3,1,00000000000000,,,,...,0,0,0,0,0,0,1,202310,202401,2240
